In [1]:
import sys
from pathlib import Path
from typing import Any

import logging
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
import yaml
from langchain.chat_models import init_chat_model

NOTEBOOK_CWD = Path.cwd()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "data").exists() else NOTEBOOK_CWD.parent
if PROJECT_ROOT != NOTEBOOK_CWD:
    sys.path = [
        path_entry for path_entry in sys.path
        if path_entry not in ("", str(NOTEBOOK_CWD))
    ]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graph.graph import build_add_train_graph
from graph.graph_state import llm_prompt, entity_output_parser, entity_schema

# 关闭 transformers 的 warning/load report
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.utils.loading_report").setLevel(logging.ERROR)

# 关闭 Loading weights / 下载进度条
hf_logging.disable_progress_bar()
disable_progress_bars()


In [2]:
add_train_graph = build_add_train_graph()

train_path = str(PROJECT_ROOT / "data/GPT_data_source/add_train_graph/train.txt")
valid_path = str(PROJECT_ROOT / "data/matscholar/valid.txt")
unlabeled_pool_path = str(
    PROJECT_ROOT / "data/GPT_data_source/excel_data/openalex_materials_abstracts_500.xlsx"
)
paper_batch_size = 100

ENV_CONFIG_FILE = PROJECT_ROOT / "env_config.yaml"


In [3]:
def load_env_config(path: Path = ENV_CONFIG_FILE) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

ENV_CONFIG = load_env_config()


In [4]:
primary_model = init_chat_model(
    model="Doubao-Seed-2.0-pro",
    model_provider="openai",
    temperature=0,
    api_key=ENV_CONFIG["openai_api_key"],
    base_url=ENV_CONFIG["openai_base_url"],
)

judge_model = init_chat_model(
    model="MiniMax-M2.7",
    model_provider="openai",
    temperature=0,
    api_key=ENV_CONFIG["openai_api_key"],
    base_url=ENV_CONFIG["openai_base_url"],
)


In [5]:
res = add_train_graph.invoke(
    {
        "train_path": train_path,
        "valid_path": valid_path,
        "unlabeled_pool_path": unlabeled_pool_path,
        "paper_batch_size": paper_batch_size,
    },
    context={
        "prompt": llm_prompt,
        "output_parser": entity_output_parser,
        "ner_model": "bert_bilstm_crf",
        "entity_schema": entity_schema,
        "llm": primary_model,
        "judge_llm": judge_model,
        "trace_enabled": True,
    },
)


[GraphTrace] -> initialize_add_train | iter=1/6 | batch=0
[GraphTrace] <- initialize_add_train | updates=['train_path', 'valid_path', 'unlabeled_pool_path', 'paper_batch_size', 'total_paper_count', 'iterations', 'iteration', 'distance_ratio_threshold', 'min_distance_ratio_threshold', 'max_distance_ratio_threshold', 'model_distance_ratio_threshold', 'threshold_step', 'current_batch', 'processed_sample_ids', 'processed_paper_ids', 'ner_bio_results', 'ner_entity_dicts', 'llm_outputs', 'model_entity_dicts', 'distance_ratio_records', 'decision_records', 'previous_metrics', 'best_metrics']
[GraphTrace] -> train_ner | iter=1/6 | batch=0
开始第1轮训练：




Epoch 1/18 - 阶段1（联合训练）
Train Loss: 15.3160
Valid Loss: 5.5670
Valid Precision: 0.7826
Valid Recall: 0.7949
Valid F1: 0.7887
保存当前最优模型: E:\大学学习\毕设\26毕设\NRE_project\graph\model\bert_bilstm_crf\bert_bilstm_crf_iter_0.pt

Epoch 2/18 - 阶段1（联合训练）
Train Loss: 4.1605
Valid Loss: 4.9847
Valid Precision: 0.8225
Valid Recall: 0.8479
Valid F1: 0.8350
保存当前最优模型:

In [8]:
print(res.get("best_model_path"))
for item in res.get('previous_metrics'):
    print(item)
print(res.get("best_metrics"))

E:\大学学习\毕设\26毕设\NRE_project\graph\model\bert_bilstm_crf\bert_bilstm_crf_iter_1.pt
{'loss': 9.923224649215667, 'precision': 0.8366315789473684, 'recall': 0.8635375923511517, 'f1': 0.8498716852010265}
{'loss': 9.816570180982291, 'precision': 0.8400498545907769, 'recall': 0.878748370273794, 'f1': 0.8589634664401019}
{'loss': 9.48656698448352, 'precision': 0.8391376451077943, 'recall': 0.8796175575836592, 'f1': 0.8589009123700403}
{'loss': 10.291032500762318, 'precision': 0.8392484342379958, 'recall': 0.8735332464146024, 'f1': 0.8560477001703578}
{'loss': 9.320887942420976, 'precision': 0.8370833333333333, 'recall': 0.8730986527596697, 'f1': 0.8547117634545841}
{'loss': 5.619600637867106, 'precision': 0.8362032759344813, 'recall': 0.8652759669708823, 'f1': 0.8504912430585221}
{'loss': 9.816570180982291, 'precision': 0.8400498545907769, 'recall': 0.878748370273794, 'f1': 0.8589634664401019}
